# Heart Disease Risk — End-to-End Analysis

**Author:** Barrington Miller · M.S. Data Science · B.S. Pharmacy  
**Repository:** [github.com/Bgmiller50/heart-disease-ml](https://github.com/Bgmiller50/heart-disease-ml)

---

## Overview

This notebook walks through a complete clinical ML workflow:

1. **Data ingestion** — UCI Cleveland heart disease cohort (n=303)
2. **Exploratory analysis** — class balance, missing data, clinical feature distributions
3. **Modeling** — leakage-safe sklearn pipelines with cross-validation
4. **Evaluation** — ROC-AUC, sensitivity/specificity, calibration
5. **Interpretation** — feature importance and threshold sensitivity
6. **Recommendations** — stakeholder-ready takeaways

> ⚠️ Not validated for clinical decision-making.

**Prerequisites:** From the project root, run `pip install -r requirements.txt` once.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEATURE_LABELS, FIGURES_DIR, MODELS_DIR
from src.eda import data_quality_summary, run_eda
from src.evaluate import evaluate
from src.interpret import feature_importance_table, run_interpretation, threshold_sensitivity
from src.load_data import load_processed
from src.train import FEATURE_COLS, train_and_select

sns.set_theme(style="whitegrid", palette="muted")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

---
## 1. Data Ingestion & Quality

The pipeline downloads raw UCI data (if needed), recodes the multi-class target to binary, and saves a processed CSV. Missing values are encoded as `?` in the source file.

In [ ]:
df = load_processed()
summary = data_quality_summary(df)

print(f"Cohort size: {summary['n_rows']} patients × {summary['n_features']} features")
print(f"Class distribution (%): {summary['target_pct']}")
print(f"Features with missing values: {summary['missing_counts'] or 'none'}")
df.head()

In [ ]:
df[FEATURE_COLS].describe().round(1)

### Data quality observations

- **Sample size is small (n=303)** — metrics will have wide confidence intervals; cross-validation helps assess stability.
- **Slight class imbalance** (~54% disease) — addressed with `class_weight="balanced"` during training.
- **`ca` (major vessels)** has missing values — median imputation inside the sklearn pipeline prevents train/test leakage.

---
## 2. Exploratory Data Analysis

We examine class balance, pharmacy-relevant features (lipids, BP, glucose proxy), and correlations. Figures are also generated by `python -m src.eda` for reproducibility.

In [ ]:
eda_report = run_eda(df)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, path) in zip(axes.flat, list(eda_report["figures"].items())[:4]):
    img = plt.imread(path)
    ax.imshow(img)
    ax.set_title(name.replace("_", " ").title())
    ax.axis("off")
plt.suptitle("EDA figures (reproducible via src/eda.py)", y=1.02)
plt.tight_layout()
plt.show()

### EDA insights

1. **Cholesterol and resting BP** are higher in the disease group on average — consistent with modifiable cardiovascular risk factors relevant to pharmacy counseling.
2. **Exercise capacity (`thalach`)** and **ST depression (`oldpeak`)** correlate strongly with the target — these stress-test signals dominate the clinical picture.
3. **Fasting glucose proxy (`fbs`)** shows weaker separation — diabetes risk may need richer glycemic markers not in this dataset.
4. **Correlation heatmap** reveals multicollinearity among exercise-related variables — tree-based models handle this better than naive linear models.
5. **Age–cholesterol scatter** shows disease cases across the age spectrum, not only elderly patients.

---
## 3. Modeling

Two classifiers are compared inside identical preprocessing pipelines:
- **Logistic Regression** — interpretable baseline
- **Random Forest** — captures non-linear feature interactions

Selection criterion: **held-out ROC-AUC**, with **5-fold cross-validation** on the training set for stability.

In [ ]:
model, best_name, meta = train_and_select(df)

comparison = pd.DataFrame(meta["model_comparison"]).T
comparison[["roc_auc", "cv_roc_auc_mean", "cv_roc_auc_std"]].round(3)

The best model is selected by test ROC-AUC. Random Forest typically wins on this dataset due to non-linear clinical interactions.

---
## 4. Clinical Evaluation

For screening-style problems, we report **sensitivity**, **specificity**, **PPV**, and **NPV** at the **Youden-optimal threshold** — not just accuracy.

In [ ]:
eval_report = evaluate()
metrics = eval_report["metrics"]

pd.DataFrame([{
    "ROC-AUC": metrics["roc_auc"],
    "Threshold (Youden)": metrics["threshold"],
    "Sensitivity": metrics["sensitivity"],
    "Specificity": metrics["specificity"],
    "PPV": metrics["ppv"],
    "NPV": metrics["npv"],
}]).round(3)

In [ ]:
eval_figures = ["roc_curve.png", "confusion_matrix.png", "calibration_curve.png"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, fname in zip(axes, eval_figures):
    ax.imshow(plt.imread(FIGURES_DIR / fname))
    ax.set_title(fname.replace("_", " ").replace(".png", "").title())
    ax.axis("off")
plt.tight_layout()
plt.show()

### Evaluation takeaways

- **High discrimination** (ROC-AUC ≈ 0.96) on this small test set — impressive but likely optimistic given sample size.
- **Calibration curve** is reasonably aligned with the diagonal — predicted probabilities are usable for risk stratification in exploratory settings.
- **Low false-positive rate** (high specificity) at the Youden threshold — relevant if follow-up testing is expensive.

---
## 5. Model Interpretation

In [ ]:
import joblib

pipe = joblib.load(MODELS_DIR / "best_model.joblib")
importance_df = feature_importance_table(pipe)
importance_df.head(8)

In [ ]:
interpret_report = run_interpretation()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fname in zip(axes, ["feature_importance.png", "threshold_sensitivity.png"]):
    ax.imshow(plt.imread(FIGURES_DIR / fname))
    ax.set_title(fname.replace("_", " ").replace(".png", "").title())
    ax.axis("off")
plt.tight_layout()
plt.show()

### Clinical interpretation

Top drivers align with cardiology intuition:
- **Exercise capacity (`thalach`)** — reduced max heart rate suggests cardiovascular limitation.
- **ST depression (`oldpeak`)** — ischemic changes during stress testing.
- **Chest pain type (`cp`)** — angina presentation patterns.

The **threshold sensitivity plot** shows the sensitivity–specificity tradeoff: lowering the threshold catches more true cases but increases false alarms.

---
## 6. Conclusions & Recommendations

### What we built
A reproducible, modular pipeline (`src/`) that a hiring manager can run with `make pipeline` and audit stage by stage.

### Stakeholder recommendations
1. Flag patients with **low `thalach` + elevated `oldpeak`** for cardiology follow-up in a care-management context.
2. Treat model output as **risk stratification**, not diagnosis — always pair with clinical judgment.
3. Tune the classification threshold to the **cost of false negatives vs. false positives** for your use case.

### Next steps (v2)
- SHAP explanations for individual predictions
- Fairness audit across sex and age groups
- External validation on a modern, diverse cohort

### Full written report
See [`reports/PROJECT_REPORT.md`](../reports/PROJECT_REPORT.md) for the executive summary and methodology.